## 1、加载数据集

In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset   

In [2]:
import pandas as pd
# 加载数据
file_path = '../data/news.csv'
data = pd.read_csv(file_path)

# 显示数据的前几行
data.head()


,label,text
0,教育,澳移民子女成长记：带着中国心融入主流社会新华网悉尼5月31日电 无论哪个国家的父辈与子女间都...
1,体育,快船vs火箭首发：休城旨在练兵 小德帕特森进先发新浪体育讯北京时间4月10日消息，在常规赛还...
2,科技,3英寸屏高清闪存DV 三洋TH1特价1499 作者：中关村在线 飘雪 ...
3,时尚,贝嫂乏味归乏味 还有人买账Victoria Beckham 乏味归乏味 还有人买账大姐大的阵...
4,房产,三亚岭南赶房集 金九银十再兴购房游纯粹的旅行闲适有余却“收获”不足，设计一条可以兼容曼妙风景...


In [3]:
from sklearn.preprocessing import LabelEncoder
# 使用LabelEncoder将标签转换为数值形式
label_encoder = LabelEncoder()
data["label_num"] = label_encoder.fit_transform(data["label"])

In [4]:
# 获取标签和数值之间的对应关系
label_mapping = dict(zip(data["label"], data["label_num"]))

print("标签到数值的映射:", label_mapping)

标签到数值的映射: {'教育': 4, '体育': 0, '科技': 8, '时尚': 5, '房产': 3, '家居': 2, '财经': 9, '时政': 6, '娱乐': 1, '游戏': 7}


In [5]:
data

,label,text,label_num
0,教育,澳移民子女成长记：带着中国心融入主流社会新华网悉尼5月31日电 无论哪个国家的父辈与子女间都...,4
1,体育,快船vs火箭首发：休城旨在练兵 小德帕特森进先发新浪体育讯北京时间4月10日消息，在常规赛还...,0
2,科技,3英寸屏高清闪存DV 三洋TH1特价1499 作者：中关村在线 飘雪 ...,8
3,时尚,贝嫂乏味归乏味 还有人买账Victoria Beckham 乏味归乏味 还有人买账大姐大的阵...,5
4,房产,三亚岭南赶房集 金九银十再兴购房游纯粹的旅行闲适有余却“收获”不足，设计一条可以兼容曼妙风景...,3
...,...,...,...
995,娱乐,组图：本-斯蒂勒与布莱克助阵《寻找伴郎》首映新浪娱乐讯 北京时间3月18日(美国当地时间3月...,1
996,房产,"房源阶段性不足价格高涨 楼市将上演新一轮疯狂近日，中海地产(企业专区,旗下楼盘)以70.06...",3
997,科技,宽屏广角高清DC！佳能110IS仅售1980【山东IT在线报道】佳能IXUS 110 IS装...,8
998,时政,公安部建成打拐DNA数据库通缉50名人贩●不到1个月，侦破拐卖儿童、妇女案件逾300起，解救...,6


In [6]:
from datasets import Dataset

dataset = Dataset.from_pandas(data)
dataset

Dataset({
    features: ['label', 'text', 'label_num'],
    num_rows: 1000
})

## 2、数据集处理

In [7]:
dataset = dataset.filter(lambda x: x["text"] is not None)
dataset

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset({
    features: ['label', 'text', 'label_num'],
    num_rows: 1000
})

In [8]:
dataset = dataset.train_test_split(test_size=0.2)     # 分类数据集可以按照比例划分
dataset

DatasetDict({
    train: Dataset({
        features: ['label', 'text', 'label_num'],
        num_rows: 800
    })
    test: Dataset({
        features: ['label', 'text', 'label_num'],
        num_rows: 200
    })
})

In [9]:
dataset["train"][0]

{'label': '娱乐',
 'text': '盖-里奇翻新《福尔摩斯》 推理探案变动作冒险当“福尔摩斯”遇上“两杆老烟枪”会怎样？答案还要问英国导演盖·里奇（Guy Ritchie），因为他最近已经接手为华纳兄弟执导一部新版的《福尔摩斯》，而且这次将和以往影视作品里的福尔摩斯很不一样。夏洛克·福尔摩斯（又译歇洛克·福尔摩斯）是英国小说家阿瑟·柯南·道尔（Arthur Conan Doyle）在19世纪末创作的一系列破案侦探题材故事的主人公，他头脑冷静、观察力敏锐，推理能力更是无人能及。此外他经常拿着烟斗与手杖，旁边总少不了好朋友华生医生。福尔摩斯是世界上被搬上银幕或荧屏次数最多的文学形象，诞生至今，已经有超过75名不同的演员曾在约200部电视或电影中演绎过这一角色，其中包括克里斯托弗·李、巴兹尔·拉思伯恩、威廉·吉列、杰雷米·布雷特等等。这次华纳的新版《福尔摩斯》（Sherlock Holmes）将根据莱昂纳尔·威格拉姆（Lionel Wigram）即将上市的同名漫画改编，在这部漫画中福尔摩斯丢掉了维多利亚时代的沉闷，成为一个击剑高手，赤手空拳制服罪犯，而故事也从推理破案变成了以动作冒险为主。编译：尼莫 ',
 'label_num': 1}

In [10]:
dataset["train"].info

DatasetInfo(description='', citation='', homepage='', license='', features={'label': Value(dtype='string', id=None), 'text': Value(dtype='string', id=None), 'label_num': Value(dtype='int32', id=None)}, post_processed=None, supervised_keys=None, task_templates=None, builder_name=None, dataset_name=None, config_name=None, version=None, splits=None, download_checksums=None, download_size=None, post_processing_size=None, dataset_size=None, size_in_bytes=None)

In [11]:
# 从HuggingFace加载，输入模型名称，即可加载对于的分词器
tokenizer = AutoTokenizer.from_pretrained('bert-base-chinese', cache_dir='./models')
tokenizer

d:\program\anaconda3\envs\transformers\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


BertTokenizerFast(name_or_path='bert-base-chinese', vocab_size=21128, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}

In [12]:
def process_function(examples):
    tokenized_examples = tokenizer(examples["text"], max_length=128, truncation=True, padding="max_length", return_tensors='pt')
    tokenized_examples["labels"] = examples["label_num"]
    return tokenized_examples

tokenized_datasets = dataset.map(process_function, batched=True, remove_columns=dataset["train"].column_names)
tokenized_datasets

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 800
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 200
    })
})

In [13]:
tokenized_datasets["train"][0]

{'input_ids': [101,
  4667,
  118,
  7027,
  1936,
  5436,
  3173,
  517,
  4886,
  2209,
  3040,
  3172,
  518,
  2972,
  4415,
  2968,
  3428,
  1359,
  1220,
  868,
  1088,
  7372,
  2496,
  100,
  4886,
  2209,
  3040,
  3172,
  100,
  6878,
  677,
  100,
  697,
  3327,
  5439,
  4170,
  3366,
  100,
  833,
  2582,
  3416,
  8043,
  5031,
  3428,
  6820,
  6206,
  7309,
  5739,
  1744,
  2193,
  4028,
  4667,
  185,
  7027,
  1936,
  8020,
  100,
  100,
  8021,
  8024,
  1728,
  711,
  800,
  3297,
  6818,
  2347,
  5307,
  2970,
  2797,
  711,
  1290,
  5287,
  1040,
  2475,
  2809,
  2193,
  671,
  6956,
  3173,
  4276,
  4638,
  517,
  4886,
  2209,
  3040,
  3172,
  518,
  8024,
  5445,
  684,
  6821,
  3613,
  2199,
  1469,
  809,
  2518,
  2512,
  6228,
  868,
  1501,
  7027,
  4638,
  4886,
  2209,
  3040,
  3172,
  2523,
  679,
  671,
  3416,
  511,
  1909,
  3821,
  1046,
  185,
  4886,
  2209,
  3040,
  3172,
  8020,
  1348,
  6406,
  3623,
  3821,
  1046,
  185,
  4886,


## 3、创建模型

In [14]:
num_class = len(data['label_num'].unique())
model = AutoModelForSequenceClassification.from_pretrained('bert-base-chinese', cache_dir='./models' ,num_labels=num_class)
model

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-chinese and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(21128, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [15]:
model.config

BertConfig {
  "_name_or_path": "bert-base-chinese",
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "directionality": "bidi",
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2",
    "3": "LABEL_3",
    "4": "LABEL_4",
    "5": "LABEL_5",
    "6": "LABEL_6",
    "7": "LABEL_7",
    "8": "LABEL_8",
    "9": "LABEL_9"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_2": 2,
    "LABEL_3": 3,
    "LABEL_4": 4,
    "LABEL_5": 5,
    "LABEL_6": 6,
    "LABEL_7": 7,
    "LABEL_8": 8,
    "LABEL_9": 9
  },
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "pooler_fc_size": 768,
  "pooler_num_attention_heads": 12,
  "pooler_num_fc_layers":

In [16]:
model.config.num_labels

10

## 4、创建评估函数

In [17]:
import evaluate

clf_metrics = evaluate.combine(["accuracy", "f1", "precision", "recall"])

Using the latest cached version of the module from C:\Users\Administrator\.cache\huggingface\modules\evaluate_modules\metrics\evaluate-metric--accuracy\f887c0aab52c2d38e1f8a215681126379eca617f96c447638f751434e8e65b14 (last modified on Sun Jun 30 17:10:21 2024) since it couldn't be found locally at evaluate-metric--accuracy, or remotely on the Hugging Face Hub.
Using the latest cached version of the module from C:\Users\Administrator\.cache\huggingface\modules\evaluate_modules\metrics\evaluate-metric--f1\0ca73f6cf92ef5a268320c697f7b940d1030f8471714bffdb6856c641b818974 (last modified on Sun Jun 30 17:10:23 2024) since it couldn't be found locally at evaluate-metric--f1, or remotely on the Hugging Face Hub.
Using the latest cached version of the module from C:\Users\Administrator\.cache\huggingface\modules\evaluate_modules\metrics\evaluate-metric--precision\4e7f439a346715f68500ce6f2be82bf3272abd3f20bdafd203a2c4f85b61dd5f (last modified on Sat Aug 31 10:14:08 2024) since it couldn't be fou

In [18]:
import evaluate

acc_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")
# f1_metric

Using the latest cached version of the module from C:\Users\Administrator\.cache\huggingface\modules\evaluate_modules\metrics\evaluate-metric--accuracy\f887c0aab52c2d38e1f8a215681126379eca617f96c447638f751434e8e65b14 (last modified on Sun Jun 30 17:10:21 2024) since it couldn't be found locally at evaluate-metric--accuracy, or remotely on the Hugging Face Hub.
Using the latest cached version of the module from C:\Users\Administrator\.cache\huggingface\modules\evaluate_modules\metrics\evaluate-metric--f1\0ca73f6cf92ef5a268320c697f7b940d1030f8471714bffdb6856c641b818974 (last modified on Sun Jun 30 17:10:23 2024) since it couldn't be found locally at evaluate-metric--f1, or remotely on the Hugging Face Hub.


In [19]:
def eval_metric(eval_predict):
    predictions, labels = eval_predict
    predictions = predictions.argmax(axis=-1)
    acc = acc_metric.compute(references=labels, predictions=predictions)
    f1 = f1_metric.compute(references=labels, predictions=predictions, average="weighted")
    acc.update(f1)
    return acc

## 5、创建训练器

In [20]:
train_args = TrainingArguments(
    output_dir="./checkpoints",      # 输出文件夹
    per_device_train_batch_size=64,  # 训练时的batch_size
    per_device_eval_batch_size=128,  # 验证时的batch_size
    logging_steps=10,                # log 打印的频率
    eval_strategy="epoch",           # 评估策略
    save_strategy="epoch",           # 保存策略
    save_total_limit=3,              # 最大保存数
    learning_rate=2e-5,              # 学习率
    weight_decay=0.01,               # weight_decay
    metric_for_best_model="f1",      # 设定评估指标
    load_best_model_at_end=True,     # 训练完成后加载最优模型
    num_train_epochs=10,             # 训练轮数
    report_to=['tensorboard']
    )    
train_args

TrainingArguments(
_n_gpu=1,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
dispatch_batches=None,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=False,
evaluation_s

## 6、训练模型

In [21]:
from transformers import DataCollatorWithPadding
trainer = Trainer(model=model,
                  args=train_args,
                  train_dataset=tokenized_datasets["train"],
                  eval_dataset=tokenized_datasets["test"],
                  data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
                  compute_metrics=eval_metric)

In [22]:
trainer.train()

  0%|          | 0/130 [00:00<?, ?it/s]

{'loss': 2.0211, 'grad_norm': 7.149596691131592, 'learning_rate': 1.8461538461538465e-05, 'epoch': 0.77}


  0%|          | 0/2 [00:00<?, ?it/s]

{'eval_loss': 1.3873578310012817, 'eval_accuracy': 0.78, 'eval_f1': 0.7495995258998779, 'eval_runtime': 1.0849, 'eval_samples_per_second': 184.349, 'eval_steps_per_second': 1.843, 'epoch': 1.0}
{'loss': 1.3641, 'grad_norm': 6.7477521896362305, 'learning_rate': 1.6923076923076924e-05, 'epoch': 1.54}


  0%|          | 0/2 [00:00<?, ?it/s]

...

  0%|          | 0/2 [00:00<?, ?it/s]

{'eval_loss': 0.12824423611164093, 'eval_accuracy': 0.97, 'eval_f1': 0.9702067207722116, 'eval_runtime': 0.5656, 'eval_samples_per_second': 353.613, 'eval_steps_per_second': 3.536, 'epoch': 9.0}
{'loss': 0.0487, 'grad_norm': 1.1735042333602905, 'learning_rate': 1.5384615384615387e-06, 'epoch': 9.23}
{'loss': 0.0465, 'grad_norm': 0.5303908586502075, 'learning_rate': 0.0, 'epoch': 10.0}


  0%|          | 0/2 [00:00<?, ?it/s]

{'eval_loss': 0.12582337856292725, 'eval_accuracy': 0.97, 'eval_f1': 0.9702067207722116, 'eval_runtime': 0.5509, 'eval_samples_per_second': 363.05, 'eval_steps_per_second': 3.63, 'epoch': 10.0}
{'train_runtime': 83.6523, 'train_samples_per_second': 95.634, 'train_steps_per_second': 1.554, 'train_loss': 0.4588154568121983, 'epoch': 10.0}


TrainOutput(global_step=130, training_loss=0.4588154568121983, metrics={'train_runtime': 83.6523, 'train_samples_per_second': 95.634, 'train_steps_per_second': 1.554, 'total_flos': 526259908608000.0, 'train_loss': 0.4588154568121983, 'epoch': 10.0})

## 7、评估

In [23]:
trainer.evaluate(tokenized_datasets["test"])

  0%|          | 0/2 [00:00<?, ?it/s]

{'eval_loss': 0.18132974207401276,
 'eval_accuracy': 0.975,
 'eval_f1': 0.9751733679982816,
 'eval_runtime': 0.56,
 'eval_samples_per_second': 357.173,
 'eval_steps_per_second': 3.572,
 'epoch': 10.0}

## 8、预测

In [24]:
trainer.predict(tokenized_datasets["test"])

  0%|          | 0/2 [00:00<?, ?it/s]

PredictionOutput(predictions=array([[-0.30052677, -0.52943146,  0.19245042, ..., -0.39983344,
         0.02780859, -0.32941964],
       [ 4.571658  , -0.2557597 , -0.8160582 , ..., -0.3211806 ,
        -0.28588054, -0.5892455 ],
       [-0.6230732 , -1.306985  ,  4.1520343 , ..., -0.76222956,
        -0.18092562, -0.44715455],
       ...,
       [-1.5901706 , -1.9178246 ,  2.348843  , ...,  0.01725204,
        -0.5317252 , -0.09291084],
       [-0.29060894,  4.3639627 , -1.5969478 , ..., -0.07263001,
        -0.2411894 , -0.76715755],
       [ 4.612513  , -0.3178277 , -0.72213113, ..., -0.36333036,
        -0.32521996, -0.5390459 ]], dtype=float32), label_ids=array([6, 0, 2, 0, 1, 3, 9, 2, 8, 9, 8, 2, 1, 5, 6, 8, 5, 2, 5, 6, 4, 0,
       9, 0, 6, 4, 5, 5, 8, 1, 4, 6, 3, 4, 7, 1, 7, 1, 5, 7, 2, 9, 1, 4,
       5, 3, 7, 8, 7, 0, 0, 1, 8, 8, 6, 3, 1, 0, 7, 9, 3, 6, 8, 9, 9, 1,
       9, 8, 1, 8, 8, 2, 3, 6, 5, 3, 7, 0, 5, 4, 2, 9, 3, 3, 1, 2, 1, 5,
       2, 2, 7, 0, 6, 5, 5, 4, 5, 6, 5, 

In [25]:
from transformers import pipeline

pipe = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0)

In [26]:
id2label_dic={}
for k,v in label_mapping.items():
    id2label_dic[v] = k

In [27]:
id2label_dic

{4: '教育',
 0: '体育',
 8: '科技',
 5: '时尚',
 3: '房产',
 2: '家居',
 9: '财经',
 6: '时政',
 1: '娱乐',
 7: '游戏'}

In [28]:
data['label'][1],data['text'][1]

('体育',
 '快船vs火箭首发：休城旨在练兵 小德帕特森进先发新浪体育讯北京时间4月10日消息，在常规赛还剩下三场的时候，火箭已经彻底的失去了进军季后赛的希望。所以今天战快船旨在练兵，再加上洛瑞一直有伤，帕特森和德拉季奇则接替小钢炮和斯科拉首发出场。以下为双方本场比赛的首发阵容：快船：威廉姆斯、戈登、穆恩、格里芬、乔丹火箭：德拉季奇、马丁、巴丁格、帕特森、海耶斯(新体)')

In [29]:
model.config.id2label = id2label_dic

In [30]:
pipe(data['text'][1])

[{'label': '体育', 'score': 0.9480794668197632}]